<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/temporal_llm_classification_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Temporal LLM Classification — AutoTherm Indoor

**Experiment:** Temporal thermal comfort classification using Large Language Models.
Instead of classifying single rows (static), the model receives a sequence of 10 sensor
readings spaced 30 seconds apart (covering ~5 minutes) and predicts the thermal comfort
label of the final frame based on the observed trend.

**Motivation:** Exploratory trajectory analysis (cold_trajectory_exploration.ipynb) showed
that a consistent downward trend in ambient and wrist skin temperature is observable in
the 5 minutes before Cold onset. This experiment tests whether an LLM can exploit that
trajectory signal for classification.

**Test protocol:** This notebook uses the HuggingFace built-in test split for AutoTherm
indoor, which differs from the subject-wise split (participants 14, 16, 20) used in all
tabular synthetic data experiments. This is intentional — the built-in split provides a
larger and more balanced test set for LLM evaluation, where per-API-call cost makes
full-dataset evaluation impractical.

**Models evaluated:**
- GPT-5.6 Terra (OpenAI, newest available at time of writing)
- GPT-4o-mini (OpenAI, used in prior static classification experiments for comparability)

**Author:** Aisha Dikko — MSc AI for Sustainable Development, UCL
**Supervisor:** Mark Colley

## 1. Install Dependencies

In [1]:
!pip install datasets openai scikit-learn -q

## 2. Imports and Data Loading

In [2]:
import pandas as pd
import numpy as np
import time
import re
from datasets import load_dataset
from openai import OpenAI
from sklearn.metrics import classification_report, f1_score

# Load AutoTherm indoor dataset from HuggingFace
# Using the built-in test split for LLM evaluation
# (see notebook header for protocol rationale)
dataset = load_dataset("kopetri/AutoTherm", "indoor")
test_df = dataset["test"].to_pandas()

# Extract participant ID from filename
def extract_participant_id(filename):
    match = re.search(r"participant_\d+", filename)
    return match.group() if match else "unknown"

test_df["participant_id"] = test_df["file_name"].apply(extract_participant_id)

# Sort by participant then timestamp — critical for time-ordered sequences
# Without this, consecutive rows would not represent consecutive time
test_df = test_df.sort_values(
    ["participant_id", "Timestamp"]
).reset_index(drop=True)

print("Test set shape:", test_df.shape)
print("\nLabel distribution:")
print(test_df["Label"].value_counts().sort_index())
print("\nTest participants:", sorted(test_df["participant_id"].unique()))

README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 29.8MB            

indoor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

indoor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.41MB            

indoor/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

Test set shape: (194829, 36)

Label distribution:
Label
-3     1162
-2    26429
-1    43053
 0    47583
 1    32489
 2    39031
 3     5082
Name: count, dtype: int64

Test participants: ['participant_12', 'participant_5']


## 3. API Setup

API key is loaded from Colab Secrets (key icon in left sidebar).
Store your key under the name `OPENAI_API_KEY`.
Never paste API keys directly in notebook code — they become visible
in version control and chat history.

In [3]:
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# Sensor features used in prompts
# Selected based on feature importance analysis (wk1 notebook)
# and trajectory analysis (cold_trajectory_exploration notebook)
KEY_FEATURES = [
    "Ambient_Temperature",
    "Wrist_Skin_Temperature",
    "Radiation-Temp",
    "Ambient_Humidity",
    "GSR",
    "Heart_Rate"
]

LABEL_NAMES = {
    -3: "Cold",
    -2: "Cool",
    -1: "Slightly Cool",
    0: "Neutral",
    1: "Slightly Warm",
    2: "Warm",
    3: "Hot"
}

# Verify API connection
test_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Reply with the word: connected"}],
    max_tokens=5
)
print("API status:", test_response.choices[0].message.content)

API status: connected


## 4. Window Construction

Each window consists of 10 sensor readings spaced 30 seconds apart,
covering approximately 5 minutes of data from a single participant.

Window spacing rationale: AutoTherm samples at ~42Hz. Consecutive rows
span ~0.024 seconds — too short for any physiological change to be visible.
The trajectory analysis showed Cold onset patterns operate on a 5-minute
timescale. Spacing frames 30 seconds apart (1,260 rows) captures this.

Two window types are constructed:
1. Regular windows — random sampling across all labels
2. Cold-targeted windows — deliberately ending at Cold labels
   (Cold is rare at 0.6% of test set; random sampling alone would miss it)

In [4]:
# Window parameters
N_FRAMES = 10          # Number of frames shown to the model
FRAME_SPACING = 1260   # Rows between frames (30 seconds at 42Hz)
TOTAL_SPAN = N_FRAMES * FRAME_SPACING  # Total rows spanned (~5 minutes)
RANDOM_SEED = 42       # For reproducibility


def build_spaced_windows(p_df, n_samples=50, seed=RANDOM_SEED):
    """Extract windows with frames spaced 30 seconds apart.

    Each window samples one row per FRAME_SPACING rows, giving
    a snapshot of sensor evolution over ~5 minutes.

    Args:
        p_df: DataFrame for one participant, sorted by timestamp
        n_samples: Number of windows to extract
        seed: Random seed for reproducibility

    Returns:
        List of DataFrames, each with N_FRAMES rows
    """
    windows = []
    max_start = len(p_df) - TOTAL_SPAN
    if max_start < 1:
        return windows

    np.random.seed(seed)
    start_positions = np.random.choice(
        max_start,
        size=min(n_samples, max_start),
        replace=False
    )
    for start in start_positions:
        indices = [start + i * FRAME_SPACING for i in range(N_FRAMES)]
        window = p_df.iloc[indices].copy()
        window["frame_number"] = range(1, N_FRAMES + 1)
        windows.append(window)
    return windows


def build_cold_targeted_windows(p_df, n_samples=20, seed=RANDOM_SEED):
    """Extract windows that end at a Cold (-3) label.

    Cold is only 0.6% of the test set. Without targeted sampling,
    random windows would almost never end in Cold, making Cold
    evaluation impossible. This function finds Cold rows and
    works backwards to extract the preceding 5-minute context.

    Args:
        p_df: DataFrame for one participant, sorted by timestamp
        n_samples: Number of Cold-ending windows to extract
        seed: Random seed for reproducibility

    Returns:
        List of DataFrames ending at Cold labels
    """
    windows = []
    cold_indices = p_df[p_df["Label"] == -3].index.tolist()
    # Only use Cold rows that have enough preceding context
    valid_cold = [i for i in cold_indices if i >= TOTAL_SPAN]
    if not valid_cold:
        return windows

    np.random.seed(seed)
    selected = np.random.choice(
        valid_cold,
        size=min(n_samples, len(valid_cold)),
        replace=False
    )
    for end_idx in selected:
        # Work backwards from Cold row to get preceding frames
        indices = [
            end_idx - (N_FRAMES - 1 - i) * FRAME_SPACING
            for i in range(N_FRAMES)
        ]
        indices = [max(0, idx) for idx in indices]
        window = p_df.iloc[indices].copy()
        window["frame_number"] = range(1, N_FRAMES + 1)
        windows.append(window)
    return windows


# Extract windows for all test participants
regular_windows = []
cold_windows = []

for participant in sorted(test_df["participant_id"].unique()):
    p_df = test_df[
        test_df["participant_id"] == participant
    ].reset_index(drop=True)

    regular_windows.extend(build_spaced_windows(p_df, n_samples=50))
    cold_windows.extend(build_cold_targeted_windows(p_df, n_samples=20))

all_windows = regular_windows + cold_windows

print(f"Regular windows:       {len(regular_windows)}")
print(f"Cold-targeted windows: {len(cold_windows)}")
print(f"Total windows:         {len(all_windows)}")
print(f"Window span:           ~{TOTAL_SPAN/42/60:.1f} minutes per window")

target_labels = [w.iloc[-1]["Label"] for w in all_windows]
print(f"\nTarget label distribution (frame 10):")
print(pd.Series(target_labels).value_counts().sort_index())

Regular windows:       100
Cold-targeted windows: 20
Total windows:         120
Window span:           ~5.0 minutes per window

Target label distribution (frame 10):
-3    22
-2    14
-1    20
 0    25
 1    21
 2    16
 3     2
Name: count, dtype: int64


## 5. Prompt Construction

The prompt presents 9 labelled frames to the model and asks it to
predict the 10th. Each frame shows the time offset from the target
(e.g. '270s ago') to make the temporal structure explicit.

In [5]:
def build_temporal_prompt(window):
    """Build a temporal classification prompt for one window.

    Shows N_FRAMES-1 labelled frames and asks the model to predict
    the label of the final frame based on the observed trend.

    Args:
        window: DataFrame with N_FRAMES rows, sorted by time

    Returns:
        Tuple of (prompt_string, true_label_int)
    """
    context_rows = window.iloc[:-1]   # Frames 1-9: shown with true labels
    target_row   = window.iloc[-1]    # Frame 10: label to predict

    sequence = ""
    for i, (_, row) in enumerate(context_rows.iterrows()):
        label    = int(row["Label"])
        time_ago = (N_FRAMES - 1 - i) * 30  # seconds before target
        sequence += (
            f"Frame {i+1} ({time_ago}s ago): "
            f"Ambient={row['Ambient_Temperature']:.1f}\u00b0C, "
            f"Wrist={row['Wrist_Skin_Temperature']:.2f}\u00b0C, "
            f"Radiation={row['Radiation-Temp']:.1f}\u00b0C, "
            f"Humidity={row['Ambient_Humidity']:.0f}%, "
            f"GSR={row['GSR']:.3f}, "
            f"HR={row['Heart_Rate']:.1f}bpm "
            f"-> {label} ({LABEL_NAMES[label]})\n"
        )

    # Target frame: features shown, label withheld
    sequence += (
        f"Frame 10 (now): "
        f"Ambient={target_row['Ambient_Temperature']:.1f}\u00b0C, "
        f"Wrist={target_row['Wrist_Skin_Temperature']:.2f}\u00b0C, "
        f"Radiation={target_row['Radiation-Temp']:.1f}\u00b0C, "
        f"Humidity={target_row['Ambient_Humidity']:.0f}%, "
        f"GSR={target_row['GSR']:.3f}, "
        f"HR={target_row['Heart_Rate']:.1f}bpm "
        f"-> ?"
    )

    prompt = (
        "You are a thermal comfort expert analysing wearable sensor data "
        "from a person in an indoor office. Readings are spaced 30 seconds "
        "apart covering the last 5 minutes.\n\n"
        "Thermal comfort scale: "
        "-3=Cold, -2=Cool, -1=Slightly Cool, 0=Neutral, "
        "1=Slightly Warm, 2=Warm, 3=Hot\n\n"
        f"{sequence}\n\n"
        "Look carefully at the trend across all 10 frames. Consider whether "
        "temperatures are rising, falling, or stable. Use the full scale "
        "including extreme values like -3 or 3 if the trend suggests it.\n\n"
        "Respond with exactly one integer only: -3, -2, -1, 0, 1, 2, or 3."
    )

    return prompt, int(target_row["Label"])


# Verify prompt on first window
sample_prompt, sample_label = build_temporal_prompt(all_windows[0])
print("Sample prompt (first window):")
print(sample_prompt)
print(f"\nTrue label: {sample_label} ({LABEL_NAMES[sample_label]})")

Sample prompt (first window):
You are a thermal comfort expert analysing wearable sensor data from a person in an indoor office. Readings are spaced 30 seconds apart covering the last 5 minutes.

Thermal comfort scale: -3=Cold, -2=Cool, -1=Slightly Cool, 0=Neutral, 1=Slightly Warm, 2=Warm, 3=Hot

Frame 1 (270s ago): Ambient=22.5°C, Wrist=35.35°C, Radiation=22.8°C, Humidity=31%, GSR=1.309, HR=80.0bpm -> -1 (Slightly Cool)
Frame 2 (240s ago): Ambient=22.4°C, Wrist=35.33°C, Radiation=22.6°C, Humidity=31%, GSR=1.251, HR=78.4bpm -> -2 (Cool)
Frame 3 (210s ago): Ambient=22.3°C, Wrist=35.33°C, Radiation=22.4°C, Humidity=32%, GSR=1.203, HR=78.4bpm -> -1 (Slightly Cool)
Frame 4 (180s ago): Ambient=22.2°C, Wrist=35.33°C, Radiation=22.2°C, Humidity=32%, GSR=1.157, HR=80.0bpm -> -1 (Slightly Cool)
Frame 5 (150s ago): Ambient=22.1°C, Wrist=35.27°C, Radiation=22.1°C, Humidity=32%, GSR=1.116, HR=78.4bpm -> -1 (Slightly Cool)
Frame 6 (120s ago): Ambient=22.0°C, Wrist=35.19°C, Radiation=21.9°C, Humidit

## 6. Experiment Runner

Shared function to run the temporal classification experiment
with any specified model. Handles API errors gracefully by
defaulting to Neutral (0) on parse failure.

In [6]:
def run_temporal_experiment(model_name, windows, use_system_prompt=False,
                             temperature=None, delay=0.3):
    """Run temporal LLM classification on all windows.

    Args:
        model_name: OpenAI model identifier string
        windows: List of window DataFrames
        use_system_prompt: Whether to add a system message
            (used for GPT-5.6 Terra which ignores temperature)
        temperature: Sampling temperature (None = model default)
            GPT-5.5 and GPT-5.6 do not support temperature=0
        delay: Seconds between API calls (avoids rate limits)

    Returns:
        Tuple of (predictions_list, actuals_list)
    """
    predictions = []
    actuals     = []

    system_message = (
        "You are a thermal comfort classification expert. "
        "You must respond with exactly one integer from -3 to 3. "
        "Never respond with anything else. "
        "Never default to 0 unless the data strongly suggests neutral comfort."
    )

    for i, window in enumerate(windows):
        try:
            prompt, actual_label = build_temporal_prompt(window)

            messages = []
            if use_system_prompt:
                messages.append({"role": "system", "content": system_message})
            messages.append({"role": "user", "content": prompt})

            # Build API call kwargs — only include temperature if specified
            # GPT-5.5 and GPT-5.6 family reject temperature != 1
            call_kwargs = {
                "model": model_name,
                "messages": messages,
                "max_completion_tokens": 10,
            }
            if temperature is not None:
                call_kwargs["temperature"] = temperature

            response = client.chat.completions.create(**call_kwargs)
            prediction_text = response.choices[0].message.content.strip()

            try:
                prediction = int(prediction_text)
                # Clamp to valid label range
                if prediction not in range(-3, 4):
                    prediction = 0
            except ValueError:
                # Model returned non-integer — default to Neutral
                prediction = 0

        except Exception as e:
            print(f"  API error on window {i}: {e}")
            prediction    = 0
            actual_label  = int(window.iloc[-1]["Label"])

        predictions.append(prediction)
        actuals.append(actual_label)

        time.sleep(delay)

        if (i + 1) % 20 == 0:
            print(f"  Processed {i+1}/{len(windows)} windows")

    return predictions, actuals


def evaluate_results(predictions, actuals, label="Experiment"):
    """Compute and print evaluation metrics for one experiment.

    Args:
        predictions: List of predicted integer labels
        actuals: List of true integer labels
        label: Name for display in output

    Returns:
        Dict with macro_f1 and cold_f1 as float values
    """
    results_df = pd.DataFrame({
        "Actual":    actuals,
        "Predicted": predictions
    })

    print(f"\n{'='*60}")
    print(f"RESULTS: {label}")
    print(f"{'='*60}")
    print(f"Unique predictions: {sorted(results_df['Predicted'].unique())}")
    print(f"\nConfusion Matrix:")
    print(pd.crosstab(
        results_df["Actual"], results_df["Predicted"],
        rownames=["Actual"], colnames=["Predicted"]
    ))
    print(f"\nClassification Report:")
    print(classification_report(
        results_df["Actual"], results_df["Predicted"],
        zero_division=0
    ))

    macro_f1 = f1_score(
        results_df["Actual"], results_df["Predicted"],
        average="macro", zero_division=0
    )
    per_class_f1 = f1_score(
        results_df["Actual"], results_df["Predicted"],
        average=None, labels=[-3, -2, -1, 0, 1, 2, 3],
        zero_division=0
    )
    cold_f1 = per_class_f1[0]

    print(f"Macro F1:  {macro_f1:.4f}")
    print(f"Cold F1:   {cold_f1:.4f}")

    return {"macro_f1": macro_f1, "cold_f1": cold_f1,
            "results_df": results_df}


print("Experiment runner and evaluator defined.")

Experiment runner and evaluator defined.


## 7. Experiment A — GPT-5.6 Terra (Temporal)

GPT-5.6 Terra is OpenAI's newest balanced model (released July 9, 2026).
It does not support temperature control (only default temperature=1 accepted).
A system prompt is used to discourage defaulting to Neutral.

Note: GPT-5.6 Terra is a reasoning-optimised model. As the results below show,
reasoning-focused models are not well-suited to direct sensor classification —
they tend to default to conservative middle-of-scale predictions regardless
of prompt engineering. This is itself an informative finding.

In [7]:
print("Running Experiment A: GPT-5.6 Terra temporal classification...")
print(f"Windows: {len(all_windows)} | Model: gpt-5.6-terra | temperature: default")

preds_terra, actuals_terra = run_temporal_experiment(
    model_name="gpt-5.6-terra",
    windows=all_windows,
    use_system_prompt=True,
    temperature=None   # GPT-5.6 does not support temperature=0
)

results_terra = evaluate_results(
    preds_terra, actuals_terra,
    label="GPT-5.6 Terra — Temporal (5-min window, 10 frames)"
)

Running Experiment A: GPT-5.6 Terra temporal classification...
Windows: 120 | Model: gpt-5.6-terra | temperature: default
  Processed 20/120 windows
  Processed 40/120 windows
  Processed 60/120 windows
  Processed 80/120 windows
  Processed 100/120 windows
  Processed 120/120 windows

RESULTS: GPT-5.6 Terra — Temporal (5-min window, 10 frames)
Unique predictions: [np.int64(0), np.int64(3)]

Confusion Matrix:
Predicted   0  3
Actual          
-3         22  0
-2         14  0
-1         20  0
 0         25  0
 1         21  0
 2         15  1
 3          1  1

Classification Report:
              precision    recall  f1-score   support

          -3       0.00      0.00      0.00        22
          -2       0.00      0.00      0.00        14
          -1       0.00      0.00      0.00        20
           0       0.21      1.00      0.35        25
           1       0.00      0.00      0.00        21
           2       0.00      0.00      0.00        16
           3       0.50      0.

## 8. Experiment B — GPT-4o-mini (Temporal)

GPT-4o-mini was used in all prior static LLM classification experiments,
providing direct comparability. It supports temperature=0 for deterministic
outputs, which is preferred for reproducible evaluation.

This experiment tests whether adding temporal trajectory context (5-minute
window) improves performance relative to the static zero-shot baseline
(macro F1 = 0.4447 from the static experiments notebook).

In [8]:
print("Running Experiment B: GPT-4o-mini temporal classification...")
print(f"Windows: {len(all_windows)} | Model: gpt-4o-mini | temperature: 0")

preds_4omini, actuals_4omini = run_temporal_experiment(
    model_name="gpt-4o-mini",
    windows=all_windows,
    use_system_prompt=False,
    temperature=0   # Deterministic — ensures reproducible results
)

results_4omini = evaluate_results(
    preds_4omini, actuals_4omini,
    label="GPT-4o-mini — Temporal (5-min window, 10 frames)"
)

Running Experiment B: GPT-4o-mini temporal classification...
Windows: 120 | Model: gpt-4o-mini | temperature: 0
  Processed 20/120 windows
  Processed 40/120 windows
  Processed 60/120 windows
  Processed 80/120 windows
  Processed 100/120 windows
  Processed 120/120 windows

RESULTS: GPT-4o-mini — Temporal (5-min window, 10 frames)
Unique predictions: [np.int64(-3), np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Confusion Matrix:
Predicted  -3  -2  -1   0   1   2   3
Actual                               
-3          0  22   0   0   0   0   0
-2          1  11   2   0   0   0   0
-1          1  15   4   0   0   0   0
 0          0   2   1  12   7   3   0
 1          0   3   0   0   5  10   3
 2          0   0   0   0   0   7   9
 3          0   0   0   0   0   0   2

Classification Report:
              precision    recall  f1-score   support

          -3       0.00      0.00      0.00        22
          -2       0.21      0.79      0.33        14
  

## 9. Save Results

In [9]:
# Save results CSVs for dissertation reporting
results_terra["results_df"].to_csv(
    "temporal_llm_gpt56terra_results.csv", index=False
)
results_4omini["results_df"].to_csv(
    "temporal_llm_gpt4omini_results.csv", index=False
)
print("Results saved.")

Results saved.


## 10. Summary and Comparison

Full comparison across all LLM experiments conducted in this project.
Static classification results are from the LLM_Thermal_Comfort_Classification notebook.

In [10]:
# Static experiment results (from LLM_Thermal_Comfort_Classification.ipynb)
# These are fixed reference values — not recomputed here
STATIC_RESULTS = {
    "GPT-4o-mini Zero-shot 3-class":  {"macro_f1": 0.4447, "cold_f1": 0.00},
    "GPT-4o-mini Few-shot 3-class":   {"macro_f1": 0.3154, "cold_f1": 0.03},
    "GPT-4o-mini Zero-shot 7-class":  {"macro_f1": 0.1552, "cold_f1": 0.00},
    "GPT-4o-mini Few-shot 7-class":   {"macro_f1": 0.3142, "cold_f1": 0.00},
}

print("=" * 70)
print("COMPLETE LLM EXPERIMENT RESULTS")
print("=" * 70)
print(f"{'Method':<42} {'Macro F1':>10} {'Cold F1':>10}")
print("-" * 70)

# Static results
for name, res in STATIC_RESULTS.items():
    print(f"{name:<42} {res['macro_f1']:>10.4f} {res['cold_f1']:>10.2f}")

print("-" * 70)

# Temporal results — live from this notebook
print(
    f"{'GPT-5.6 Terra Temporal':<42} "
    f"{results_terra['macro_f1']:>10.4f} "
    f"{results_terra['cold_f1']:>10.2f}"
)
print(
    f"{'GPT-4o-mini Temporal':<42} "
    f"{results_4omini['macro_f1']:>10.4f} "
    f"{results_4omini['cold_f1']:>10.2f}"
)

print("=" * 70)
print("\nKey findings:")
print(f"  Cold F1 = 0.00 across all LLM conditions (static and temporal)")
print(f"  GPT-5.6 Terra collapses to 1-3 classes despite being newest model")
print(f"  GPT-4o-mini temporal uses full 7-class scale — more suitable for classification")
print(f"  Temporal context did not improve Cold detection over static classification")
print(f"  Between-subject generalisation gap confirmed across all LLM protocols")

COMPLETE LLM EXPERIMENT RESULTS
Method                                       Macro F1    Cold F1
----------------------------------------------------------------------
GPT-4o-mini Zero-shot 3-class                  0.4447       0.00
GPT-4o-mini Few-shot 3-class                   0.3154       0.03
GPT-4o-mini Zero-shot 7-class                  0.1552       0.00
GPT-4o-mini Few-shot 7-class                   0.3142       0.00
----------------------------------------------------------------------
GPT-5.6 Terra Temporal                         0.1214       0.00
GPT-4o-mini Temporal                           0.3165       0.00

Key findings:
  Cold F1 = 0.00 across all LLM conditions (static and temporal)
  GPT-5.6 Terra collapses to 1-3 classes despite being newest model
  GPT-4o-mini temporal uses full 7-class scale — more suitable for classification
  Temporal context did not improve Cold detection over static classification
  Between-subject generalisation gap confirmed across all LLM pr